# UNet Models

This notebook covers the UNet building blocks and full models. UNets are the backbone of most modern diffusion models -- `ConditionedUNet` directly serves as the `epsilon_theta` noise predictor in DDPM/DDIM pipelines, while `UNet` covers all supervised spatial prediction tasks (segmentation, denoising, super-resolution).

## Class hierarchy and integration map

```
SinusoidalTimeEmbedding          used inside: TimeEmbeddingMLP
  \-- TimeEmbeddingMLP           used inside: ConditioningBuilder
        \-- ConditioningBuilder  used inside: ConditionedUNet.forward()

ConvBlock            (from convolution.py)
  \-- ConditionedConvBlock       used as block_cls inside UNetStage / UNetUpStage

SpatialAttentionBlock            used inside: UNetStage.attention, UNetUpStage.stage.attention

UNetStage                        used inside: BaseUNet (encoder path of UNet / ConditionedUNet)
  \-- UNetUpStage                used inside: BaseUNet (decoder path of UNet / ConditionedUNet)
        \-- UNet                 complete plain U-Net
        \-- ConditionedUNet      complete FiLM-conditioned U-Net
```

## Task selection guide

| Task | Recommended class |
|---|---|
| Image / volume segmentation (no conditioning) | `UNet` |
| Signal / image denoising (no conditioning) | `UNet` |
| Diffusion noise predictor (DDPM/DDIM backbone) | `ConditionedUNet(time_conditioning=True)` |
| Class-conditional generation | `ConditionedUNet(num_classes=N)` |
| Text-to-image / text-guided generation | `ConditionedUNet(cross_attention_dim=D, attention_type="cross")` |
| Building a custom encoder-decoder with skip connections | `UNetStage` + `UNetUpStage` manually |
| Spatial feature map with long-range context | `UNet(attention_downsample_factors=[...])` |
| Conditioned audio waveform synthesis | `ConditionedUNet(conv_dim=1, time_conditioning=True)` |

**Key distinction -- UNet vs ConvNet:** `ConvNet` progressively downsamples and discards spatial resolution (encoder only). `UNet` restores the original spatial size through skip connections -- use it whenever the output must be the same size as the input.

All models work across `conv_dim=1/2/3`:

| `conv_dim` | Input shape | Real data |
|---|---|---|
| 1 | `(B, C, L)` | audio waveform, ECG, time series |
| 2 | `(B, C, H, W)` | images, spectrograms, 2D medical imaging |
| 3 | `(B, C, D, H, W)` | CT/MRI volume, voxel grid, video |

In [1]:
import torch

from ml_suite.models.unet import (
    UNet, ConditionedUNet,
    SpatialAttentionBlock,
    SinusoidalTimeEmbedding, TimeEmbeddingMLP, ConditioningBuilder,
    UNetStage, UNetUpStage,
)
from ml_suite.models.convolution import ConvBlock, ConditionedConvBlock
import matplotlib.pyplot as plt

---
## 1. Building Blocks

These modules form the conditioning pipeline and spatial processing primitives that `UNet` and `ConditionedUNet` assemble internally. Use them standalone only when building a custom architecture -- for end-to-end usage, go straight to Section 2 or 3.

**How to pick:**
- Need a scalar timestep -> high-dim vector? `SinusoidalTimeEmbedding` (fixed) or `TimeEmbeddingMLP` (learnable projection on top)
- Need to combine time + class + context into one FiLM vector? `ConditioningBuilder`
- Need spatial self- or cross-attention at a specific feature map? `SpatialAttentionBlock`
- Need an encoder stage with skip outputs? `UNetStage`
- Need a decoder stage that merges a skip? `UNetUpStage`

### 1.1 `SinusoidalTimeEmbedding`

Maps a scalar timestep `(B,)` to a fixed sinusoidal vector `(B, embedding_dim)`. No learnable parameters.

**Integrates into:** `TimeEmbeddingMLP` -- this is the first stage of the conditioning pipeline. `ConditionedUNet` never calls `SinusoidalTimeEmbedding` directly; it uses `TimeEmbeddingMLP` inside `ConditioningBuilder`.

**Use standalone when:** you need a raw sinusoidal time feature for an external use -- e.g. as a fixed Fourier feature for a neural field, or as an input to a custom conditioning MLP not managed by `ConditioningBuilder`.

**Use for:** any diffusion / score model that encodes a scalar timestep; neural fields with positional Fourier features; custom conditioning pipelines.

In [2]:
sin_emb = SinusoidalTimeEmbedding(embedding_dim=256)

# Real: DDPM timestep t ~ {1, ..., 1000}, normalised -> t/T ~ (0, 1]
time = torch.linspace(0.0, 1.0, 4)   # (B,)
emb  = sin_emb(time)                  # (B, embedding_dim)
print(f"SinusoidalTimeEmbedding: {time.shape} -> {emb.shape}")   # (4, 256)

SinusoidalTimeEmbedding: torch.Size([4]) -> torch.Size([4, 256])


### 1.2 `TimeEmbeddingMLP`

Applies a 2-layer MLP on top of sinusoidal or learned time encoding, projecting to `embedding_dim`.

**Integrates into:** `ConditioningBuilder` -- it is the time branch. `ConditionedUNet` uses `ConditioningBuilder`, which uses `TimeEmbeddingMLP` internally.

**Use standalone when:** you need a standalone time embedding that outputs a vector you will wire into a custom conditioning pipeline (e.g. adding it to a per-block context outside of `ConditionedUNet`).

**Use for:** DDPM / DDIM noise predictors, score-based generative models, any model that conditions on a diffusion timestep.

| `embedding_type` | When to use |
|---|---|
| `"sinusoidal"` | Fixed frequency basis -- no extra parameters, works for normalised `t/T` |
| `"learned"` | Learnable base embedding -- preferred when timestep distribution is non-uniform |

In [3]:
# Real: DDPM timestep t ~ {1..1000}, normalised -> t/T ~ (0, 1]
time = torch.rand(4)   # (B,)

t_sin = TimeEmbeddingMLP(embedding_dim=128, embedding_type="sinusoidal")
print(f"Sinusoidal TimeEmbeddingMLP: {t_sin(time).shape}")   # (4, 128)

t_learned = TimeEmbeddingMLP(embedding_dim=128, embedding_type="learned")
print(f"Learned TimeEmbeddingMLP:    {t_learned(time).shape}")   # (4, 128)

Sinusoidal TimeEmbeddingMLP: torch.Size([4, 128])
Learned TimeEmbeddingMLP:    torch.Size([4, 128])


### 1.3 `ConditioningBuilder`

Combines time + class label + global context into a single FiLM conditioning vector `(B, condition_dim)` by summing active branches.

**Integrates into:** `ConditionedUNet.forward()` -- the first thing `ConditionedUNet` does is call `self.conditioning(...)` to build this vector, then pass it to every `ConditionedConvBlock` via FiLM.

**Use standalone when:** assembling a custom conditioned architecture with `UNetStage` + `UNetUpStage` where you want the same conditioning logic but manage the network structure yourself.

**Use for:** any architecture that uses FiLM-style conditioning (scale + shift applied per-channel after normalisation).

In [4]:
builder = ConditioningBuilder(
    condition_dim=128,
    time_conditioning=True,
    num_classes=10,
    class_dropout_prob=0.1,   # classifier-free guidance: null class with 10% prob during training
    global_context_dim=64,
)

B = 2
cond = builder(
    batch_size=B, device=torch.device("cpu"), dtype=torch.float32,
    time=torch.rand(B),                      # normalised timestep t/T ~ (0, 1]
    class_labels=torch.randint(0, 10, (B,)),
    global_context=torch.randn(B, 64),       # e.g. CLIP pooled embedding
)
print(f"ConditioningBuilder output: {cond.shape}")   # (2, 128) -- single FiLM vector per sample

ConditioningBuilder output: torch.Size([2, 128])


### 1.4 `SpatialAttentionBlock`

Attention over a spatial feature map `(B, C, *spatial)`. Internally flattens spatial dims to tokens, applies attention, then reshapes back.

**Integrates into:** `UNetStage.attention` and `UNetUpStage.stage.attention`. Both `UNet` and `ConditionedUNet` automatically construct and insert `SpatialAttentionBlock` instances at the stages specified by `attention_downsample_factors`.

**Use standalone when:** you want to add spatial attention at a custom location in a non-standard architecture -- e.g. after a bottleneck in a custom encoder, or between two `ConvNet` stages.

**Use for:** global spatial context in denoising U-Nets, text-guided spatial generation (cross-attention), feature fusion at the bottleneck of an encoder-decoder.

| `attention_type` | Use for |
|---|---|
| `"self"` | Global spatial context, long-range dependencies (plain UNet) |
| `"cross"` | Attend to external token sequence -- text-conditioned spatial generation |
| `"self_cross"` | Both -- local structure via self-attn + external guidance via cross-attn |

In [5]:
# (batch, channels, height, width)
x_2d = torch.randn(2, 64, 16, 16)

# Self-attention -- global spatial context
sa = SpatialAttentionBlock(channels=64, attention_type="self", num_heads=4)
print(f"Self-attn 2D: {x_2d.shape} -> {sa(x_2d).shape}")

# Works on 1D too  -- (batch, channels, length)
sa_1d = SpatialAttentionBlock(channels=64, attention_type="self", num_heads=4)
x_1d  = torch.randn(2, 64, 32)
print(f"Self-attn 1D: {x_1d.shape} -> {sa_1d(x_1d).shape}")

# Cross-attention -- spatial features attend to external context tokens
# Real: x_2d ~ UNet bottleneck (B, C, H, W), cross_ctx ~ CLIP text tokens (B, 77, 512)
ca        = SpatialAttentionBlock(channels=64, attention_type="cross", num_heads=4, cross_attention_dim=128)
cross_ctx = torch.randn(2, 20, 128)   # (B, n_tokens, context_dim)
print(f"Cross-attn 2D: x={x_2d.shape}, ctx={cross_ctx.shape} -> {ca(x_2d, cross_context=cross_ctx).shape}")

# Self + cross -- long-range spatial context then external guidance
sca      = SpatialAttentionBlock(channels=64, attention_type="self_cross", num_heads=4, cross_attention_dim=128)
ctx_mask = torch.ones(2, 20, dtype=torch.bool)
print(f"Self+Cross-attn: {sca(x_2d, cross_context=cross_ctx, cross_context_mask=ctx_mask).shape}")

Self-attn 2D: torch.Size([2, 64, 16, 16]) -> torch.Size([2, 64, 16, 16])
Self-attn 1D: torch.Size([2, 64, 32]) -> torch.Size([2, 64, 32])
Cross-attn 2D: x=torch.Size([2, 64, 16, 16]), ctx=torch.Size([2, 20, 128]) -> torch.Size([2, 64, 16, 16])
Self+Cross-attn: torch.Size([2, 64, 16, 16])


### 1.5 `UNetStage` -- encoder stage

A stack of `ConvBlock` (or `ConditionedConvBlock`) modules with optional strided downsampling on the first block and an optional `SpatialAttentionBlock` applied after the convolutions.

**Integrates into:** `BaseUNet` encoder path (shared by `UNet` and `ConditionedUNet`).

**Use standalone when:** building a custom encoder where you want skip-connection-compatible stages without committing to the full UNet decoder structure -- e.g., a custom encoder for a diffusion model that uses a different decoder or bottleneck.

**Use for:** custom encoder-decoder architectures, multi-scale feature extractors, any task that needs spatially downsampled feature maps with configurable conditioning.

**Do not use directly** if you just want a complete UNet -- `UNet` and `ConditionedUNet` wire these automatically.

In [6]:
# Plain stage (UNet-style) -- no conditioning
stage = UNetStage(
    input_channels=32, output_channels=64, conv_dim=2, num_blocks=2,
    block_cls=ConvBlock, first_stride=2, norm_type="batch",
)
x   = torch.randn(2, 32, 32, 32)   # (B, C_in, H, W)
out = stage(x)
print(f"Plain UNetStage (stride=2): {x.shape} -> {out.shape}")   # (2, 64, 16, 16)

# Conditioned stage (ConditionedUNet-style) -- FiLM conditioning per block
cstage = UNetStage(
    input_channels=64, output_channels=128, conv_dim=2, num_blocks=2,
    block_cls=ConditionedConvBlock, condition_dim=128,
    first_stride=2, norm_type="group", num_groups=32,
)
x   = torch.randn(2, 64, 32, 32)
ctx = torch.randn(2, 128)   # FiLM conditioning vector from ConditioningBuilder
print(f"Conditioned UNetStage: {x.shape}, cond={ctx.shape} -> {cstage(x, condition=ctx).shape}")

# Stage with self-attention -- long-range spatial dependencies at low resolution
sa     = SpatialAttentionBlock(channels=64, attention_type="self", num_heads=4)
astage = UNetStage(
    input_channels=32, output_channels=64, conv_dim=2, num_blocks=2,
    first_stride=2, attention=sa,
)
x   = torch.randn(2, 32, 32, 32)
out = astage(x)
print(f"UNetStage + self-attn: {x.shape} -> {out.shape}")

Plain UNetStage (stride=2): torch.Size([2, 32, 32, 32]) -> torch.Size([2, 64, 16, 16])
Conditioned UNetStage: torch.Size([2, 64, 32, 32]), cond=torch.Size([2, 128]) -> torch.Size([2, 128, 16, 16])
UNetStage + self-attn: torch.Size([2, 32, 32, 32]) -> torch.Size([2, 64, 16, 16])


### 1.6 `UNetUpStage` -- decoder stage

Upsample + merge skip connection + run a `UNetStage`.

**Integrates into:** `BaseUNet` decoder path (shared by `UNet` and `ConditionedUNet`).

**Use standalone when:** building a custom decoder that must match a specific encoder's skip connection structure -- e.g. when the encoder is a pretrained `ConvNet` and you want to add a UNet-style decoder on top of it.

**Use for:** custom segmentation decoders, super-resolution upsampling paths, any spatial reconstruction task requiring skip connections.

| Option | When to use |
|---|---|
| `upsample_mode="interpolate"` | Simpler, fewer params, avoids checkerboard artefacts |
| `upsample_mode="transpose"` | Learnable upsampling; useful when output quality matters more than speed |
| `skip_mode="concat"` | Maximum information from skip; doubles channels at merge -- more compute |
| `skip_mode="add"` | Lightweight; requires matching channels; cleaner gradient path |

In [7]:
# (B, C_dec, H_low, W_low) decoder input + (B, C_skip, H_high, W_high) skip
x_dec  = torch.randn(2, 128, 8, 8)
x_skip = torch.randn(2, 64, 16, 16)

# Interpolate + concat (default) -- no learnable upsample params, keeps both streams
up = UNetUpStage(
    input_channels=128, skip_channels=64, output_channels=64,
    conv_dim=2, num_blocks=2, upsample_mode="interpolate", skip_mode="concat",
)
print(f"interp+concat: dec={x_dec.shape}, skip={x_skip.shape} -> {up(x_dec, skip=x_skip).shape}")

# Transpose + concat -- learnable upsample, more parameters
up_t = UNetUpStage(
    input_channels=128, skip_channels=64, output_channels=64,
    conv_dim=2, num_blocks=2, upsample_mode="transpose", skip_mode="concat",
)
print(f"transpose+concat: {up_t(x_dec, skip=x_skip).shape}")

# Interpolate + add -- lightweight, requires matching channels
x_dec2  = torch.randn(2, 64, 8, 8)
x_skip2 = torch.randn(2, 64, 16, 16)
up_add = UNetUpStage(
    input_channels=64, skip_channels=64, output_channels=64,
    conv_dim=2, num_blocks=2, upsample_mode="interpolate", skip_mode="add",
)
print(f"interp+add: {up_add(x_dec2, skip=x_skip2).shape}")

interp+concat: dec=torch.Size([2, 128, 8, 8]), skip=torch.Size([2, 64, 16, 16]) -> torch.Size([2, 64, 16, 16])
transpose+concat: torch.Size([2, 64, 16, 16])
interp+add: torch.Size([2, 64, 16, 16])


---
## 2. `UNet` -- full plain U-Net

A complete encoder-bottleneck-decoder pipeline. Output is the same spatial size as input. Use it when there is no external conditioning signal -- the output depends only on the input tensor.

**How it is built internally:** `UNet` passes `block_cls=ConvBlock` and `condition_dim=None` to `BaseUNet`, which assembles `UNetStage` (encoder) and `UNetUpStage` (decoder) instances.

**When to use `UNet` vs `ConditionedUNet`:** use `UNet` when there is no external conditioning signal. Use `ConditionedUNet` when the model should behave differently based on timestep, class, or text.

**How to pick `conv_dim`:**

| `conv_dim` | Input shape | Real task |
|---|---|---|
| 1 | `(B, C, L)` | waveform denoising, ECG segmentation, 1D anomaly detection |
| 2 | `(B, C, H, W)` | medical image segmentation, image denoising, depth estimation |
| 3 | `(B, C, D, H, W)` | tumour / organ segmentation in CT/MRI, volumetric reconstruction |

### 2.1 1D UNet

Operates on `(B, C, L)` tensors -- length is preserved by the encoder-decoder path.

**Integrates into:** standalone model; not used as a sub-module by any other class.

**Use standalone when:** your input and output are equal-length 1D sequences and no conditioning is needed.

**Use for:** waveform denoising, temporal event segmentation, 1D anomaly detection, EEG/ECG signal-to-signal prediction.

In [8]:
unet_1d = UNet(
    conv_dim=1, in_channels=1, out_channels=1,
    stage_channels=[32, 64, 128], blocks_per_stage=2, norm_type="batch",
)

# Real: noisy audio (B, 1, 16000),  ECG (B, 12, 5000)
x   = torch.randn(2, 1, 256)   # (B, C, L)
out = unet_1d(x)
print(f"1D UNet: {x.shape} -> {out.shape}")   # output length == input length

1D UNet: torch.Size([2, 1, 256]) -> torch.Size([2, 1, 256])


### 2.2 2D UNet

Operates on `(B, C, H, W)` tensors -- spatial resolution is preserved end-to-end.

**Integrates into:** standalone model; not used as a sub-module by any other class.

**Use standalone when:** your input and output are equal-resolution 2D images / feature maps and no conditioning is needed.

**Use for:** medical image segmentation (histology, MRI slice, CT), image denoising, depth estimation, semantic segmentation.

In [9]:
unet_2d = UNet(
    conv_dim=2, in_channels=3, out_channels=1,   # binary mask output
    stage_channels=[64, 128, 256, 512], blocks_per_stage=2,
    norm_type="group", num_groups=32,
    output_activation=None,   # logits -- apply sigmoid externally for BCELoss
)

# Real: histology (B, 3, 512, 512),  CT slice (B, 1, 512, 512)
x   = torch.randn(2, 3, 64, 64)   # (B, C, H, W)
out = unet_2d(x)
print(f"2D UNet: {x.shape} -> {out.shape}")   # spatial dims preserved

2D UNet: torch.Size([2, 3, 64, 64]) -> torch.Size([2, 1, 64, 64])


### 2.3 3D UNet

Operates on `(B, C, D, H, W)` tensors -- all three spatial dimensions are preserved.

**Integrates into:** standalone model; not used as a sub-module by any other class.

**Use standalone when:** your input is a volumetric tensor and no conditioning is needed.

**Use for:** 3D medical image segmentation (tumour, organ), volumetric reconstruction, shape completion.

In [10]:
unet_3d = UNet(
    conv_dim=3, in_channels=1, out_channels=3,   # 3-class segmentation
    stage_channels=[16, 32, 64], blocks_per_stage=2,
    norm_type="group", num_groups=16,
)

# Real: brain MRI (B, 1, 155, 240, 240)
x   = torch.randn(1, 1, 16, 16, 16)   # (B, C, D, H, W)
out = unet_3d(x)
print(f"3D UNet: {x.shape} -> {out.shape}")   # all spatial dims preserved

3D UNet: torch.Size([1, 1, 16, 16, 16]) -> torch.Size([1, 3, 16, 16, 16])


### 2.4 Skip modes: `concat` vs `add`

Affects how encoder skip connections are merged with decoder features at each stage.

| `skip_mode` | Notes | Best for |
|---|---|---|
| `"concat"` (default) | More expressive -- the decoder sees both streams independently before a conv | Dense prediction where fine detail matters |
| `"add"` | Fewer params -- the streams are summed; requires compatible channel counts | Lightweight models, residual-style decoders |

In [11]:
x = torch.randn(2, 1, 64, 64)   # (B, C, H, W)

for skip in ["concat", "add"]:
    unet = UNet(
        conv_dim=2, in_channels=1, out_channels=1,
        stage_channels=[32, 64, 128], skip_mode=skip, norm_type="batch",
    )
    n = sum(p.numel() for p in unet.parameters())
    print(f"skip_mode='{skip}': {unet(x).shape}, params={n:,}")

skip_mode='concat': torch.Size([2, 1, 64, 64]), params=472,257
skip_mode='add': torch.Size([2, 1, 64, 64]), params=390,433


### 2.5 Upsample modes: `interpolate` vs `transpose`

| `upsample_mode` | Notes | Best for |
|---|---|---|
| `"interpolate"` (default) | Nearest-neighbour, no learnable params; avoids checkerboard artefacts | Safe default for most tasks |
| `"transpose"` | Learned upsampling via `ConvTransposeNd`; adds parameters | Higher-quality outputs when generation fidelity matters |

In [12]:
x = torch.randn(2, 1, 64, 64)   # (B, C, H, W)

for up in ["interpolate", "transpose"]:
    unet = UNet(
        conv_dim=2, in_channels=1, out_channels=1,
        stage_channels=[32, 64, 128], upsample_mode=up, norm_type="batch",
    )
    n = sum(p.numel() for p in unet.parameters())
    print(f"upsample_mode='{up}': {unet(x).shape}, params={n:,}")

upsample_mode='interpolate': torch.Size([2, 1, 64, 64]), params=472,257
upsample_mode='transpose': torch.Size([2, 1, 64, 64]), params=467,233


### 2.6 Attention in the UNet

`attention_downsample_factors` controls which encoder stages receive a `SpatialAttentionBlock`, identified by their factor of spatial downsampling relative to the input.

For `stage_channels=[64, 128, 256, 512]`:
- Stage 0 (stem): factor = 1 (no downsampling)
- Stage 1: factor = 2
- Stage 2: factor = 4
- Stage 3 / bottleneck: factor = 8

`attention_locations` (`"encoder"`, `"decoder"`, `"both"`, `"bottleneck"`) controls which side of the symmetric UNet gets attention blocks.

**Rule of thumb:** add attention at stages with small spatial size (factor >= 4) -- cheap to compute and captures the global structure the network needs most.

| `attention_locations` | Use for |
|---|---|
| `"bottleneck"` | Minimal compute -- global context only at the lowest resolution |
| `"encoder"` | Encode global context; decoder uses only local skip info |
| `"decoder"` | Reconstruct with global context; encoder is purely local |
| `"both"` | Maximum expressiveness -- used in most diffusion U-Nets |

In [13]:
x = torch.randn(2, 1, 64, 64)   # (B, C, H, W)

# No attention -- pure convolutional baseline
unet_plain = UNet(
    conv_dim=2, in_channels=1, out_channels=1,
    stage_channels=[32, 64, 128], attention_downsample_factors=(),
)
n = sum(p.numel() for p in unet_plain.parameters())
print(f"No attention: {unet_plain(x).shape}, params={n:,}")

# Attention at 4x downsampled stage -- recommended starting point for diffusion models
unet_attn = UNet(
    conv_dim=2, in_channels=1, out_channels=1,
    stage_channels=[32, 64, 128],
    attention_downsample_factors=[4], attention_locations="both", num_attention_heads=4,
)
n = sum(p.numel() for p in unet_attn.parameters())
print(f"Attn at 4x: {unet_attn(x).shape}, params={n:,}")

# Attention at 2x and 4x -- richer long-range context, more compute
unet_multi = UNet(
    conv_dim=2, in_channels=1, out_channels=1,
    stage_channels=[32, 64, 128],
    attention_downsample_factors=[2, 4], attention_locations="both", num_attention_heads=4,
)
n = sum(p.numel() for p in unet_multi.parameters())
print(f"Attn at 2x+4x: {unet_multi(x).shape}, params={n:,}")

No attention: torch.Size([2, 1, 64, 64]), params=472,257
Attn at 4x: torch.Size([2, 1, 64, 64]), params=670,529
Attn at 2x+4x: torch.Size([2, 1, 64, 64]), params=770,497


---
## 3. `ConditionedUNet`

Extends `UNet` in two ways:
1. Replaces all `ConvBlock` stages with `ConditionedConvBlock` (FiLM) -- every conv block receives the global conditioning vector
2. Adds a `ConditioningBuilder` that combines time + class + global context into that vector

Cross-attention conditioning (`cross_context`) is handled by `SpatialAttentionBlock` instances at the specified stages -- it does not go through FiLM.

**`condition_dim` default:** `4 x stage_channels[0]`. Override with `condition_dim=` if you want a different width.

**How to pick conditioning sources:**

| Conditioning | When to use |
|---|---|
| `time_conditioning=True` | Diffusion noise predictor -- model must behave differently at each timestep |
| `num_classes=N` | Class-conditional generation; set `class_dropout_prob > 0` for classifier-free guidance |
| `global_context_dim=D` | Pooled embedding guidance (CLIP, speaker, style) -- scalar global FiLM |
| `cross_attention_dim=D` + `attention_downsample_factors` | Token-level guidance (text, structured sequences) -- spatially variable |

### 3.1 Time conditioning -- diffusion noise predictor

Conditions every `ConditionedConvBlock` on a scalar timestep via FiLM. This is the standard backbone for DDPM / DDIM.

**Integrates into:** standalone model -- the recommended entry point for diffusion noise prediction.

**Use standalone when:** you need `epsilon_theta(x_t, t)` or `v_theta(x_t, t)` without class or text conditioning.

**Use for:** DDPM / DDIM noise estimator `epsilon_theta(x_t, t)`, score-based model `s_theta(x_t, t)`, image denoising with learned timestep schedule.

In [14]:
unet = ConditionedUNet(
    conv_dim=2, in_channels=3, out_channels=3,
    stage_channels=[64, 128, 256], blocks_per_stage=2,
    norm_type="group", num_groups=32,
    time_conditioning=True, time_embedding_type="sinusoidal",
)

# Real: noisy RGB image x_t ~ (B, 3, 256, 256) at timestep t ~ {1, ..., 1000}
x    = torch.randn(2, 3, 64, 64)   # (B, C, H, W)
time = torch.rand(2)               # normalised t/T ~ (0, 1]
out  = unet(x, time=time)
print(f"Diffusion predictor: {x.shape}, t={time.shape} -> {out.shape}")   # (2, 3, 64, 64)

Diffusion predictor: torch.Size([2, 3, 64, 64]), t=torch.Size([2]) -> torch.Size([2, 3, 64, 64])


### 3.2 Class conditioning

Adds a learned class embedding to the FiLM conditioning vector. Enables conditional generation where the model output depends on a discrete class label.

**Integrates into:** standalone model -- combine with `time_conditioning=True` for the standard class-conditional diffusion setup.

**Use standalone when:** you want class-conditional generation without time conditioning (e.g. conditional image restoration).

`class_dropout_prob > 0` implements the unconditional branch for **classifier-free guidance** -- during training, class labels are randomly nulled so the model learns both conditional and unconditional distributions.

**Use for:** class-conditional image synthesis (ImageNet-conditioned generation), label-guided anomaly simulation.

In [15]:
unet = ConditionedUNet(
    conv_dim=2, in_channels=1, out_channels=1,
    stage_channels=[32, 64, 128],
    num_classes=10, class_dropout_prob=0.1,   # classifier-free guidance: null class with 10% prob
    norm_type="group", num_groups=32,
)

# Real: class-conditional DDPM -- x ~ (B, 1, 256, 256), labels ~ {0..9}
x      = torch.randn(2, 1, 64, 64)    # (B, C, H, W)
labels = torch.randint(0, 10, (2,))
out    = unet(x, class_labels=labels)
print(f"Class-conditioned: {x.shape}, labels={labels.tolist()} -> {out.shape}")

Class-conditioned: torch.Size([2, 1, 64, 64]), labels=[0, 2] -> torch.Size([2, 1, 64, 64])


### 3.3 Global context conditioning

A global feature vector is projected to `condition_dim` and added to the FiLM conditioning vector. Used alongside or instead of time/class.

**Integrates into:** standalone model -- combine with `time_conditioning=True` for time + global-context conditioned diffusion.

**Use standalone when:** the conditioning signal is a pooled embedding (not token-level), e.g. a speaker embedding in speech synthesis or a style vector in style transfer.

**Use for:** CLIP-guided generation (pooled text embedding), audio-to-image (audio embedding -> condition), style transfer (style vector from a style encoder).

In [16]:
unet = ConditionedUNet(
    conv_dim=2, in_channels=3, out_channels=3,
    stage_channels=[64, 128, 256],
    global_context_dim=512,   # e.g. CLIP pooled text embedding dim
    norm_type="group", num_groups=32,
)

# Real: CLIP-guided image synthesis -- ctx ~ CLIP pooled text (B, 512)
x    = torch.randn(2, 3, 64, 64)    # (B, C, H, W)
clip = torch.randn(2, 512)          # (B, embed_dim)
out  = unet(x, global_context=clip)
print(f"Global context: {x.shape}, ctx={clip.shape} -> {out.shape}")

Global context: torch.Size([2, 3, 64, 64]), ctx=torch.Size([2, 512]) -> torch.Size([2, 3, 64, 64])


### 3.4 Cross-attention conditioning

Adds `SpatialAttentionBlock` with `attention_type="cross"` at specified stages. Spatial feature map tokens attend to the external context sequence -- more expressive than global FiLM because different spatial locations can attend to different parts of the context.

**Integrates into:** standalone model -- combine with `time_conditioning=True` for the full Stable Diffusion-style backbone.

**Use standalone when:** you want spatial features to attend to a variable-length token sequence (text, audio tokens) without full time conditioning.

**Use for:** Stable Diffusion-style text-to-image (spatial features x text tokens), audio-guided video prediction.

> Cross-attention requires `attention_downsample_factors` to be non-empty -- otherwise there are no attention blocks and `cross_context` is silently unused.

In [17]:
unet = ConditionedUNet(
    conv_dim=2, in_channels=3, out_channels=3,
    stage_channels=[64, 128, 256],
    norm_type="group", num_groups=32,
    attention_downsample_factors=[4],
    attention_type="cross",
    cross_attention_dim=128,
    num_attention_heads=4,
)

# Real: CLIP/BERT text tokens (B, 77, 768) attend to spatial features
x        = torch.randn(2, 3, 64, 64)     # (B, C, H, W)
ctx_tok  = torch.randn(2, 20, 128)       # (B, n_tokens, context_dim)
ctx_mask = torch.ones(2, 20, dtype=torch.bool)
out = unet(x, cross_context=ctx_tok, cross_context_mask=ctx_mask)
print(f"Cross-attn: {x.shape}, ctx={ctx_tok.shape} -> {out.shape}")

Cross-attn: torch.Size([2, 3, 64, 64]), ctx=torch.Size([2, 20, 128]) -> torch.Size([2, 3, 64, 64])


### 3.5 Self + cross attention

`attention_type="self_cross"` adds self-attention (global spatial context) then cross-attention (external guidance) in each attention block.

**Integrates into:** standalone model -- the richest single-stage conditioning mode.

**Use standalone when:** you want the spatial feature map to first attend to itself (resolve occlusion, build global context) before attending to the external tokens.

**When to prefer this over `"cross"` alone:** when the spatial features benefit from attending to each other first (e.g. to disambiguate occlusion or texture) before incorporating text guidance.

**Use for:** Stable Diffusion-style UNet backbones (the original implementation uses self + cross at every attention block).

In [18]:
unet = ConditionedUNet(
    conv_dim=2, in_channels=3, out_channels=3,
    stage_channels=[64, 128, 256],
    norm_type="group", num_groups=32,
    attention_downsample_factors=[4],
    attention_type="self_cross", cross_attention_dim=128, num_attention_heads=4,
)
# Real: Stable Diffusion UNet -- self-attn resolves spatial context, cross-attn injects text
x       = torch.randn(2, 3, 64, 64)    # (B, C, H, W)
ctx_tok = torch.randn(2, 20, 128)      # (B, n_tokens, context_dim)
out     = unet(x, cross_context=ctx_tok)
print(f"Self+Cross UNet: {out.shape}")   # (2, 3, 64, 64)

Self+Cross UNet: torch.Size([2, 3, 64, 64])


### 3.6 Full conditioning -- all four sources

FiLM (time + class + global context) + token cross-attention can be active simultaneously.

**Integrates into:** standalone model -- the maximally expressive configuration.

**Use for:** Stable Diffusion-style backbone -- timestep drives the overall denoising schedule (FiLM), text tokens guide local spatial decisions (cross-attention), class label optionally shifts the style (FiLM).

In [19]:
unet = ConditionedUNet(
    conv_dim=2, in_channels=3, out_channels=3,
    stage_channels=[64, 128, 256, 512], blocks_per_stage=2,
    norm_type="group", num_groups=32,
    time_conditioning=True,
    num_classes=100, class_dropout_prob=0.1,   # classifier-free guidance: null class with 10% prob
    global_context_dim=512,
    attention_downsample_factors=[4, 8],
    attention_type="self_cross", attention_locations="both",
    cross_attention_dim=128, num_attention_heads=4,
)

B = 2
x            = torch.randn(B, 3, 64, 64)       # (B, C, H, W)
time         = torch.rand(B)                   # normalised t/T ~ (0, 1]
class_labels = torch.randint(0, 100, (B,))
global_ctx   = torch.randn(B, 512)             # e.g. CLIP pooled text (B, 512)
cross_ctx    = torch.randn(B, 20, 128)         # (B, n_tokens, context_dim)
ctx_mask     = torch.ones(B, 20, dtype=torch.bool)

out = unet(
    x, time=time, class_labels=class_labels,
    global_context=global_ctx,
    cross_context=cross_ctx, cross_context_mask=ctx_mask,
)
print(f"Full ConditionedUNet: {x.shape} -> {out.shape}")
print(f"Total parameters: {sum(p.numel() for p in unet.parameters()):,}")

Full ConditionedUNet: torch.Size([2, 3, 64, 64]) -> torch.Size([2, 3, 64, 64])
Total parameters: 15,705,475


### 3.7 1D ConditionedUNet -- conditioned waveform model

Operates on `(B, C, L)` waveforms with FiLM conditioning from any combination of time, class, and global context.

**Integrates into:** standalone model -- the standard backbone for 1D diffusion.

**Use standalone when:** your input is a 1D waveform or time series and you need conditional generation or denoising.

**Use for:** DiffWave-style audio diffusion, speaker-conditioned speech denoising, conditioned music generation.

In [20]:
unet_1d = ConditionedUNet(
    conv_dim=1, in_channels=1, out_channels=1,
    stage_channels=[32, 64, 128],
    norm_type="group", num_groups=32,
    time_conditioning=True,
    global_context_dim=64,   # e.g. speaker embedding from a speaker encoder
)

# Real: noisy speech waveform x_t ~ (B, 1, 16000) at timestep t
x    = torch.randn(2, 1, 256)   # (B, C, L)
time = torch.rand(2)            # normalised t/T ~ (0, 1]
spkr = torch.randn(2, 64)       # d-vector / x-vector speaker embedding
out  = unet_1d(x, time=time, global_context=spkr)
print(f"1D ConditionedUNet: {x.shape} -> {out.shape}")   # (2, 1, 256)

1D ConditionedUNet: torch.Size([2, 1, 256]) -> torch.Size([2, 1, 256])


### 3.8 3D ConditionedUNet -- volumetric diffusion

Operates on `(B, C, D, H, W)` volumes with FiLM conditioning.

**Integrates into:** standalone model -- the standard backbone for 3D conditional generation.

**Use standalone when:** your input is a volumetric tensor and you need time-conditioned or class-conditioned generation / denoising.

**Use for:** 3D medical image generation / inpainting, molecule conformation generation, voxel-based shape synthesis.

In [21]:
unet_3d = ConditionedUNet(
    conv_dim=3, in_channels=1, out_channels=1,
    stage_channels=[16, 32, 64],
    norm_type="group", num_groups=16,
    time_conditioning=True,
)

# Real: noisy MRI volume x_t ~ (B, 1, 128, 128, 128) at timestep t
x    = torch.randn(1, 1, 16, 16, 16)   # (B, C, D, H, W)
time = torch.rand(1)                   # normalised t/T ~ (0, 1]
out  = unet_3d(x, time=time)
print(f"3D ConditionedUNet: {x.shape} -> {out.shape}")   # all spatial dims preserved

3D ConditionedUNet: torch.Size([1, 1, 16, 16, 16]) -> torch.Size([1, 1, 16, 16, 16])
